# Atlas integrated scoring

This notebook combines fish use, habitat limitations, biological vulnerability, and population priorities into **relative prioritization indices** for each BSR. It calculates Levels 1 and 2; it does not evaluate individual projects or predict changes in fish abundance.

| Level | Question | What the score represents |
|---|---|---|
| **1. Integrated risk** | Where do fish use and biological vulnerability overlap with impaired habitat conditions? | The sum of life-stage fish use × population priority × limiting-factor condition × vulnerability across pathways. |
| **2. Action alignment** | Which action types are most strongly associated with the limiting factors contributing to that risk? | Existing limiting-factor risk weighted by each action's relationship to the factor. |

The basic unit is one **BSR × species × life stage × limiting factor** pathway. For example, Chinook migration and predation form one pathway within a BSR. The notebook calculates each pathway's risk contribution and sums it by life stage, species, limiting factor, and BSR.

**How to read the results:** a larger risk score indicates more overlap among the scored inputs. A larger action-alignment index indicates stronger correspondence between an action and the factors contributing to risk. Neither score is a probability, a physical habitat quantity, or a predicted restoration benefit.

**Run order:** load inputs → validate inputs → transform scores → calculate Levels 1 and 2 → validate calculations → write and verify temporary files → replace the output folder. No published output is replaced until all numerical and file checks pass.

**Source review is separate from numerical QC.** Source BSR crosswalk statuses and condition and vulnerability review flags remain visible even when every mathematical check passes. Numerical QC does not independently verify those source judgments.

See the companion [`framework.md`](framework.md) for a concise reference to the scoring framework, equations, established output fields, and interpretation limits.

## 1. Inputs

Put the five processed CSVs and polygon GeoPackage in `data/inputs`:

- `Fish Use Scores.csv`
- `Limiting factor scores.csv`
- `Vulnerability table.csv`
- `Population scores.csv`
- `LFAT.csv`
- `bsr.gpkg`

Set `INPUT_DIR_OVERRIDE` only if automatic discovery finds the wrong folder.

### Original source files

The original workbooks and supporting source table are in the selected input directory under `Original Excel/`. Use them to review source formulas, ratings, and notes. The scoring code reads the processed CSVs listed above, not the workbooks directly.

| Processed scoring input | Original source file |
|---|---|
| `Fish Use Scores.csv`; `Population scores.csv` | `Original Excel/Fish Use Score calculator - Normalized.xlsx` |
| `Vulnerability table.csv` | `Original Excel/Combined - Lifestage to Limiting Factor Crosswalk Table.xlsx` |
| `LFAT.csv` | `Original Excel/LFAT Atlas Action-LimFact Crosswalk scoring working.xlsx` |
| `Limiting factor scores.csv` | `Original Excel/BSR_LF_cell_stats.csv` |

In [1]:
from contextlib import closing
from pathlib import Path
from uuid import uuid4
import hashlib
import shutil
import sqlite3
import json
import warnings
from datetime import datetime, timezone
from tempfile import TemporaryDirectory

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        if isinstance(obj, pd.DataFrame):
            print(obj.to_string(index=False))
        else:
            print(obj)


# Optional: replace None with a folder path if automatic discovery is not appropriate.
INPUT_DIR_OVERRIDE = None

FRAMEWORK_VERSION = "2026-09-16.1"
RUN_CREATED_UTC = datetime.now(timezone.utc).isoformat()
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid4().hex[:8]
qc_records = []
calculation_qc_passed = False
worked_example_qc_passed = False


def check(condition, name, detail="", stage="input"):
    """Record a passed check or stop immediately with an actionable error."""
    if not bool(condition):
        raise ValueError(f"{stage} QC failed: {name}. {detail}".strip())
    qc_records.append({"stage": stage, "check": name, "result": "passed"})


def compare_frames(actual, expected, name, stage="calculation"):
    """Compare values and columns, allowing only normal serialization rounding."""
    try:
        pd.testing.assert_frame_equal(
            actual.reset_index(drop=True), expected.reset_index(drop=True),
            check_dtype=False, check_exact=False, rtol=1e-12, atol=1e-12,
        )
    except AssertionError as error:
        raise ValueError(f"{stage} QC failed: {name}. {error}") from error
    check(True, name, stage=stage)


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

INPUT_STEMS = {
    "fish_use": "Fish Use Scores",
    "lfat": "LFAT",
    "limiting_factor": "Limiting factor scores",
    "population": "Population scores",
    "vulnerability": "Vulnerability table",
}
FISH_USE_COLUMNS = [
    "bsr", "basin", "bsr_crosswalk_status",
    "species", "life_stage", "LS_corrected_score",
    "species_aggregate_score", "fish_use_score_decimal",
]
BSR_INPUT_FILE = "bsr.gpkg"
BSR_OUTPUT_FILE = "bsr_scores.gpkg"


def select_input_file(folder, stem, suffix=".csv"):
    # Return one exact or parenthetically numbered input, or None.
    exact = folder / f"{stem}{suffix}"
    if exact.exists():
        return exact
    matches = sorted(folder.glob(f"{stem}(*){suffix}"))
    return matches[0] if len(matches) == 1 else None


def candidate_input_directories(start):
    seen = set()
    for folder in (start, *start.parents):
        for candidate in (
            folder / "data" / "inputs",
            folder / "upload",
            folder,
        ):
            resolved = candidate.resolve()
            if resolved not in seen:
                seen.add(resolved)
                yield resolved


def locate_inputs(start, override=None):
    candidates = [Path(override).expanduser().resolve()] if override else list(
        candidate_input_directories(start)
    )
    for folder in candidates:
        if not folder.is_dir():
            continue
        selected = {
            key: select_input_file(folder, stem)
            for key, stem in INPUT_STEMS.items()
        }
        spatial_path = select_input_file(folder, "bsr", ".gpkg")
        if (
            all(path is not None for path in selected.values())
            and spatial_path is not None
        ):
            return folder, selected, spatial_path
    expected = [
        *(f"{stem}.csv" for stem in INPUT_STEMS.values()),
        BSR_INPUT_FILE,
    ]
    raise FileNotFoundError(
        "Could not find one complete input set. Expected: "
        + ", ".join(expected)
    )


INPUT_DIR, INPUT_PATHS, BSR_INPUT_PATH = locate_inputs(
    Path.cwd().resolve(), INPUT_DIR_OVERRIDE
)
if INPUT_DIR.name == "inputs" and INPUT_DIR.parent.name == "data":
    REPO_ROOT = INPUT_DIR.parent.parent
else:
    REPO_ROOT = Path.cwd().resolve()

OUTPUT_DIR = REPO_ROOT / "data" / "outputs"
QC_DIR = OUTPUT_DIR / "QC"
BSR_OUTPUT_PATH = OUTPUT_DIR / BSR_OUTPUT_FILE

raw = {
    key: pd.read_csv(
        path, usecols=FISH_USE_COLUMNS if key == "fish_use" else None
    )
    for key, path in INPUT_PATHS.items()
}

input_file_hashes = {
    key: file_sha256(path) for key, path in INPUT_PATHS.items()
}
input_file_hashes["spatial"] = file_sha256(BSR_INPUT_PATH)

input_summary = pd.DataFrame(
    [
        {
            "dataset": key,
            "file": path.name,
            "rows": len(raw[key]),
            "columns": len(raw[key].columns),
            "sha256": input_file_hashes[key],
        }
        for key, path in INPUT_PATHS.items()
    ]
)

print(f"Input directory: {INPUT_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"QC directory: {QC_DIR}")
print(f"Spatial input: {BSR_INPUT_PATH.name}")
display(input_summary)

Input directory: C:\Users\AlexThornton-Dunwood\OneDrive - Lichen Land & Water\Documents\GitHub\atlas_lvl1-2_scoring_webapp\data\inputs
Output directory: C:\Users\AlexThornton-Dunwood\OneDrive - Lichen Land & Water\Documents\GitHub\atlas_lvl1-2_scoring_webapp\data\outputs
QC directory: C:\Users\AlexThornton-Dunwood\OneDrive - Lichen Land & Water\Documents\GitHub\atlas_lvl1-2_scoring_webapp\data\outputs\QC
Spatial input: bsr.gpkg


,dataset,file,rows,columns,sha256
0,fish_use,Fish Use Scores.csv,290,8,f21d305d9b894eebc25aa16aba1a6a8aacb81229c6933c...
1,lfat,LFAT.csv,165,19,d24f71894f1ff92e6a06fa499f46ce323086b2a704ead2...
2,limiting_factor,Limiting factor scores.csv,435,18,d3bea17c18238b7e32072de49d6d81ca9028d6de9af7fd...
3,population,Population scores.csv,20,4,e8d353404cc1513a72368d7e09ee0d582cc7ab1da8f6c9...
4,vulnerability,Vulnerability table.csv,180,14,f4218b99f14e7c4884e638b01501044c49f55cd4d9d971...


## 2. Validate source data before scoring

These checks establish that the input tables can be joined without missing records, accidental duplication, or inconsistent identifiers. Each BSR must contain all 10 species/life-stage combinations and all 15 limiting factors; each action must have a row for every limiting factor, including zero-weight relationships.

The notebook reads only the designated fish-use fields. In particular, overall fish use comes from `fish_use_score_decimal`. Other fish-use score columns in that input are ignored. Required columns may be reordered; unrelated columns in the other CSVs are allowed.

Population priorities are checked within **basin × species**. They distribute weight among a species' life stages; they do not assign explicit relative priorities between species. Source priorities are used as supplied. A total of 0.99 is accepted as rounding and is reported rather than automatically renormalized.

In [2]:
EXPECTED_COLUMNS = {
    "fish_use": FISH_USE_COLUMNS,
    "lfat": [
        "action_id", "action_type", "action_definition",
        "source_action_label", "limiting_factor",
        "limiting_factor_occurrence", "directness_code",
        "directness_rating", "directness_value", "frequency_code",
        "frequency_rating", "frequency_value", "lfat_score",
        "source_sheet", "source_row", "source_directness_cell",
        "source_frequency_cell", "source_score_cell", "source_notes_cell",
    ],
    "limiting_factor": [
        "bsr", "limiting_factor", "n", "mean_r", "median_r",
        "geo_mean_r", "sd_r", "var_r", "iqr_r", "min_r", "max_r",
        "range_r", "flag_spread", "flag_low_n", "any_flag",
        "lf_condition_score_raw_1_5", "lf_condition_score",
        "condition_transformation",
    ],
    "population": [
        "basin", "species", "life_stage", "population_priority",
    ],
    "vulnerability": [
        "species", "source_life_stage", "life_stage", "limiting_factor",
        "vulnerability_rank", "vulnerability_score", "uncertainty_flag",
        "review_flag", "review_reason", "source_sheet",
        "source_rank_cell", "source_rating_cell", "source_notes_cell",
        "source_uncertainty_cell",
    ],
}

# This fixed list lets later checks confirm complete 15-factor coverage.
CANONICAL_LF = [
    "Anthropogenic Barriers",
    "Riparian Condition",
    "Floodplain Condition",
    "Side Channel and Wetland Habitat",
    "Channel and Habitat Structure",
    "Decreased Water Quantity",
    "Altered Flow Timing",
    "Decreased Sediment Quantity",
    "Increased Sediment Quantity",
    "Summer Water Temperature",
    "Winter Water Temperature",
    "Water Quality",
    "Predation",
    "Altered Primary Productivity",
    "Non-Native Species Interactions and Competition",
]


# Check column names without requiring a particular CSV column order.
schema_rows = []
for name, table in raw.items():
    missing = sorted(set(EXPECTED_COLUMNS[name]) - set(table.columns))
    check(not missing, f"{name}: required columns", f"Missing: {missing}")
    schema_rows.append({"dataset": name, "required_columns": len(EXPECTED_COLUMNS[name]), "schema_pass": True})
schema_qc = pd.DataFrame(schema_rows)

key_columns = {
    "fish_use": ["bsr", "basin", "species", "life_stage", "bsr_crosswalk_status"],
    "population": ["basin", "species", "life_stage"],
    "limiting_factor": ["bsr", "limiting_factor"],
    "vulnerability": ["species", "source_life_stage", "life_stage", "limiting_factor"],
    "lfat": ["action_id", "action_type", "action_definition", "limiting_factor"],
}
for name, columns in key_columns.items():
    for column in columns:
        values = raw[name][column]
        check(values.notna().all(), f"{name}.{column}: no missing identifiers")
        if column != "action_id":
            raw[name][column] = values.astype(str).str.strip()
            check(raw[name][column].ne("").all(), f"{name}.{column}: no blank identifiers")

# Convert only fields used numerically, and reject blanks, text, and infinities.
numeric_columns = {
    "fish_use": ["LS_corrected_score", "species_aggregate_score", "fish_use_score_decimal"],
    "population": ["population_priority"],
    "limiting_factor": ["lf_condition_score_raw_1_5", "lf_condition_score", "n"],
    "vulnerability": ["vulnerability_rank", "vulnerability_score"],
    "lfat": ["directness_value", "frequency_value", "lfat_score"],
}
for name, columns in numeric_columns.items():
    for column in columns:
        values = pd.to_numeric(raw[name][column], errors="coerce")
        check(np.isfinite(values).all(), f"{name}.{column}: finite numeric values")
        raw[name][column] = values

unique_keys = {
    "fish_use": ["bsr", "species", "life_stage"],
    "population": ["basin", "species", "life_stage"],
    "limiting_factor": ["bsr", "limiting_factor"],
    "vulnerability": ["species", "source_life_stage", "limiting_factor"],
    "lfat": ["action_id", "limiting_factor"],
}
for name, keys in unique_keys.items():
    duplicates = raw[name].loc[raw[name].duplicated(keys, keep=False), keys]
    check(duplicates.empty, f"{name}: unique scoring keys", duplicates.head(10).to_dict("records"))

fish = raw["fish_use"]
check(fish["fish_use_score_decimal"].between(0, 1).all(), "Overall fish use is 0–1")
check(fish["LS_corrected_score"].ge(0).all(), "Source life-stage fish use is nonnegative")
check(fish["LS_corrected_score"].max() > 0, "At least one positive life-stage fish-use score")
check(fish["species_aggregate_score"].ge(0).all(), "Species fish use is nonnegative")
for column in ["basin", "fish_use_score_decimal", "bsr_crosswalk_status"]:
    check(fish.groupby("bsr")[column].nunique().eq(1).all(), f"One {column} value per BSR")
check(fish.groupby(["bsr", "species"])["species_aggregate_score"].nunique().eq(1).all(), "One species fish-use value per BSR/species")

for name in ["limiting_factor", "vulnerability", "lfat"]:
    check(set(raw[name]["limiting_factor"]) == set(CANONICAL_LF), f"{name}: canonical limiting factors")
for name, keys in [("limiting_factor", ["bsr"]), ("vulnerability", ["species", "source_life_stage"]), ("lfat", ["action_id"])]:
    check(raw[name].groupby(keys)["limiting_factor"].nunique().eq(15).all(), f"{name}: complete 15-factor coverage")
check(set(fish["bsr"]) == set(raw["limiting_factor"]["bsr"]), "Fish-use and condition BSR coverage agrees")

stage_keys = set(map(tuple, fish[["species", "life_stage"]].drop_duplicates().to_numpy()))
for bsr, group in fish.groupby("bsr"):
    check(set(map(tuple, group[["species", "life_stage"]].to_numpy())) == stage_keys, f"{bsr}: complete species/life-stage coverage")
population_keys = ["basin", "species", "life_stage"]
check(
    set(map(tuple, fish[population_keys].drop_duplicates().to_numpy()))
    == set(map(tuple, raw["population"][population_keys].to_numpy())),
    "Fish-use and population-priority keys agree",
)
check(
    stage_keys == set(map(tuple, raw["vulnerability"][["species", "life_stage"]].drop_duplicates().to_numpy())),
    "Fish-use and vulnerability life stages agree",
)

check(raw["population"]["population_priority"].between(0, 1).all(), "Population priorities are 0–1")
population_review = raw["population"].groupby(["basin", "species"], as_index=False).agg(
    priority_sum=("population_priority", "sum"), life_stage_count=("life_stage", "size")
)
population_review["difference_from_one"] = population_review["priority_sum"] - 1.0
check(np.allclose(population_review["priority_sum"], 1.0, rtol=0, atol=0.011), "Population-priority sums allow only source rounding", population_review.to_dict("records"))
check(raw["limiting_factor"]["lf_condition_score_raw_1_5"].between(1, 5).all(), "Raw condition ratings are 1–5")
check(raw["vulnerability"]["vulnerability_rank"].between(1, 15).all(), "Vulnerability ranks are 1–15")
check(raw["vulnerability"]["vulnerability_rank"].mod(1).eq(0).all(), "Vulnerability ranks are integers")
for column in ["directness_value", "frequency_value", "lfat_score"]:
    check(raw["lfat"][column].between(0, 1).all(), f"LFAT {column} is 0–1")
check(np.allclose(raw["lfat"]["lfat_score"], raw["lfat"]["directness_value"] * raw["lfat"]["frequency_value"], rtol=1e-9, atol=1e-12), "LFAT weight equals directness × frequency")
for column in ["action_type", "action_definition"]:
    check(raw["lfat"].groupby("action_id")[column].nunique().eq(1).all(), f"One {column} per action ID")
check(raw["lfat"].groupby("action_type")["action_id"].nunique().eq(1).all(), "Action types identify unique action IDs")

# Verify expected migration mapping rather than silently collapsing other stages.
for (species, life_stage), group in raw["vulnerability"].groupby(["species", "life_stage"]):
    stages = set(group["source_life_stage"])
    if life_stage == "Migration":
        check(stages == {"Adult Migration & Holding", "Juvenile Emigration"}, f"{species}: adult/juvenile migration source coverage")
    else:
        check(len(stages) == 1, f"{species}/{life_stage}: one uncombined source stage")

display(schema_qc)
display(population_review)

,dataset,required_columns,schema_pass
0,fish_use,8,True
1,lfat,19,True
2,limiting_factor,18,True
3,population,4,True
4,vulnerability,14,True


,basin,species,priority_sum,life_stage_count,difference_from_one
0,Catherine Creek,Bull Trout,1.00,2,0.00
1,Catherine Creek,Chinook,0.99,4,-0.01
2,Catherine Creek,Steelhead,1.00,4,0.00
3,Upper Grande Ronde,Bull Trout,1.00,2,0.00
4,Upper Grande Ronde,Chinook,1.00,4,0.00
5,Upper Grande Ronde,Steelhead,1.00,4,0.00


## 3. Put the inputs on their scoring scales

Four inputs enter Level 1. A fifth, the action relationship weight, is used in Level 2.

| Input | Calculation used by the notebook | Meaning of a larger value | Max. Value
|---|---|---|---|
| Life-stage fish use | $\text{life-stage fish use} = \dfrac{\text{source life-stage fish use}}{\text{maximum source life-stage fish use}}$ | More fish use on the source index | 1 |
| Limiting-factor condition | $\text{condition score} = 0.01 + (\text{raw rating} - 1) \times \dfrac{0.99}{4}$ | Greater impairment, assuming the source rubric has this direction | 1 |
| Biological vulnerability | $\text{vulnerability score} = 1 - (\text{rank} - 1) \times \dfrac{0.99}{14}$ | Greater vulnerability to that limiting factor | 1 |
| Population priority | $\text{population priority} = \text{source priority}$ | More weight for this life stage within its basin and species | 1 |
| Action relationship weight | $\text{action relationship weight} = \text{directness} \times \text{frequency}$ | A stronger action and limiting-factor relationship | 1 |

**The 0.01 floor:** the least impaired condition and the lowest vulnerability retain a small nonzero contribution. Zero life-stage fish use still produces zero impact and risk. Linear rank conversion and the nonzero floor are modeling choices, not quantities established by the QC checks.

### Fish-use variables

| Output field | Treatment | Role |
|---|---|---|
| `LS_corrected_score_source` | Original life-stage value | Audit trail |
| `LS_corrected_score` | Source divided by the recorded maximum; 0–1 | The fish-use multiplier in impact and risk |
| `species_aggregate_score` | Source value retained; may exceed 1 | Context only |
| `fish_use_score` | Read exclusively from `fish_use_score_decimal`; 0–1 | Overall BSR context only |


Population priorities remain as supplied, including accepted rounding differences. No separate between-species priority multiplier is introduced.

In [3]:
# Preserve the source life-stage score and rename BSR fish use for stable
# output naming. BSR fish use is retained for context only.
fish_use = raw["fish_use"].rename(
    columns={
        "LS_corrected_score": "LS_corrected_score_source",
        "fish_use_score_decimal": "fish_use_score",
    }
).copy()

# Normalize life-stage fish use by the maximum value across the complete
# input table. Dividing by the maximum preserves meaningful zeros and the
# ratios among source scores while constraining the multiplier to 0 to 1.
life_stage_fish_use_source_max = fish_use["LS_corrected_score_source"].max()
if (
    not np.isfinite(life_stage_fish_use_source_max)
    or life_stage_fish_use_source_max <= 0
):
    raise ValueError(
        "Life-stage fish-use scores must include at least one positive value."
    )
fish_use["LS_corrected_score"] = (
    fish_use["LS_corrected_score_source"]
    / life_stage_fish_use_source_max
)

# Copy population priorities so the imported table is not modified.
population = raw["population"].copy()

# Preserve the CSV's condition score and give the raw rating a concise name.
condition = raw["limiting_factor"].rename(
    columns={
        "lf_condition_score_raw_1_5": "condition_score_raw_1_5",
        "lf_condition_score": "condition_score_source",
    }
).copy()

# Convert condition linearly: source rating 1 becomes 0.01 and 5 becomes 1.0.
condition["condition_score"] = (
    0.01
    + (condition["condition_score_raw_1_5"] - 1.0) * (0.99 / 4.0)
)

# Preserve the CSV's vulnerability score before recalculating it from rank.
vulnerability = raw["vulnerability"].rename(
    columns={"vulnerability_score": "vulnerability_score_source"}
).copy()

# Convert vulnerability linearly: rank 1 becomes 1.0 and rank 15 becomes 0.01.
vulnerability["vulnerability_score"] = (
    1.0
    - (vulnerability["vulnerability_rank"] - 1.0) * (0.99 / 14.0)
)

# Copy the action crosswalk so derived calculations do not modify the import.
lfat = raw["lfat"].copy()


fish_use["life_stage_fish_use_normalization_max"] = life_stage_fish_use_source_max

# Preserve the source BSR correspondence status without a user-entered override.
identifier_review = fish_use[["bsr", "basin", "bsr_crosswalk_status"]].drop_duplicates().copy()
identifier_review["bsr_match_review_required"] = (
    identifier_review["bsr_crosswalk_status"].ne("exact_identifier")
)
identifier_review["input_review_status"] = np.where(
    identifier_review["bsr_match_review_required"],
    "Provisional: BSR correspondence requires review",
    "Exact BSR identifier",
)
identifier_review = identifier_review.sort_values(["basin", "bsr"]).reset_index(drop=True)
fish_use = fish_use.merge(
    identifier_review.drop(columns=["basin", "bsr_crosswalk_status"]),
    on="bsr", how="left", validate="many_to_one",
)

normalization_summary = pd.DataFrame([{
    "run_id": RUN_ID,
    "method": "global maximum across complete fish-use input table",
    "source_field": "LS_corrected_score",
    "source_output_field": "LS_corrected_score_source",
    "normalized_output_field": "LS_corrected_score",
    "denominator": float(life_stage_fish_use_source_max),
    "source_min": float(fish_use["LS_corrected_score_source"].min()),
    "normalized_min": float(fish_use["LS_corrected_score"].min()),
    "normalized_max": float(fish_use["LS_corrected_score"].max()),
    "source_rows": len(fish_use),
}])
maximum_rows = fish_use.loc[
    fish_use["LS_corrected_score_source"].eq(life_stage_fish_use_source_max),
    ["bsr", "species", "life_stage", "LS_corrected_score_source"],
]

check(fish_use["LS_corrected_score"].between(0, 1).all(), "Normalized life-stage fish use is 0–1", stage="transformation")
check(np.isclose(fish_use["LS_corrected_score"].max(), 1), "Maximum normalized fish use equals 1", stage="transformation")
check(condition["condition_score"].between(0.01, 1).all(), "Transformed condition is 0.01–1", stage="transformation")
check(vulnerability["vulnerability_score"].between(0.01, 1).all(), "Transformed vulnerability is 0.01–1", stage="transformation")
check(np.allclose(0.01 + (np.array([1., 5.]) - 1) * 0.99 / 4, [0.01, 1.0]), "Condition endpoint anchors", stage="transformation")
check(np.allclose(1 - (np.array([1., 15.]) - 1) * 0.99 / 14, [1.0, 0.01]), "Vulnerability endpoint anchors", stage="transformation")

assumptions = pd.DataFrame([
    ["Life-stage fish use", "LS_corrected_score", "Source divided by the maximum across the complete input table; source and denominator retained."],
    ["Species fish use", "species_aggregate_score", "Unchanged context field; may exceed 1; not a direct multiplier."],
    ["Overall fish use", "fish_use_score_decimal", "Input mapped to output fish_use_score; context only."],
    ["Population priority", "population_priority", "Within-species life-stage weights; source rounding retained; no between-species weighting."],
    ["Condition direction", "condition_score", "Assumes raw 1 means least impairment and 5 greatest impairment; verify against source rubric."],
    ["Vulnerability", "vulnerability_score", "Rank 1 maps to 1 and rank 15 to 0.01 using a linear transformation."],
    ["Migration", "maximum vulnerability_score", "Use the larger adult/juvenile vulnerability separately for each species and limiting factor."],
    ["Action weight", "lfat_score", "Directness × frequency; source product checked before scoring."],
    ["Action alignment", "overall_benefit_score", "Sum across actions; can count a limiting-factor contribution multiple times; not predicted benefit."],
], columns=["component", "field_or_rule", "implementation"])

print(f"Life-stage fish-use normalization denominator: {life_stage_fish_use_source_max:g}")
display(maximum_rows)
display(identifier_review)
unresolved_bsrs = identifier_review.loc[identifier_review["bsr_match_review_required"], "bsr"].tolist()
if unresolved_bsrs:
    print("Source BSR correspondence is provisional: " + ", ".join(unresolved_bsrs))

Life-stage fish-use normalization denominator: 3.59


,bsr,species,life_stage,LS_corrected_score_source
88,CC9,Bull Trout,Spawning & Resident,3.59


,bsr,basin,bsr_crosswalk_status,bsr_match_review_required,input_review_status
0,CC1,Catherine Creek,provisional_positional,True,Provisional: BSR correspondence requires review
1,CC2,Catherine Creek,provisional_positional,True,Provisional: BSR correspondence requires review
2,CC3,Catherine Creek,provisional_positional,True,Provisional: BSR correspondence requires review
3,CC4,Catherine Creek,provisional_positional,True,Provisional: BSR correspondence requires review
4,CC5,Catherine Creek,provisional_positional,True,Provisional: BSR correspondence requires review
5,CC6,Catherine Creek,provisional_positional,True,Provisional: BSR correspondence requires review
6,CC7,Catherine Creek,provisional_positional,True,Provisional: BSR correspondence requires review
7,CC8,Catherine Creek,provisional_positional,True,Provisional: BSR correspondence requires review
8,CC9,Catherine Creek,provisional_positional,True,Provisional: BSR correspondence requires review
9,UGR1,Upper Grande Ronde,exact_identifier,False,Exact BSR identifier


Source BSR correspondence is provisional: CC1, CC2, CC3, CC4, CC5, CC6, CC7, CC8, CC9


## 4. Align migration vulnerability

Fish use and population priorities contain one `Migration` life stage for Chinook and Steelhead. Vulnerability contains separate adult and juvenile migration records. For each species and limiting factor, use the larger of the two vulnerability scores:

$$\text{migration vulnerability} = \max(\text{adult migration vulnerability},\ \text{juvenile migration vulnerability})$$

For example, adult vulnerability 0.8 and juvenile vulnerability 0.4 produce a combined score of 0.8. This represents the more vulnerable migration pathway without adding migration twice. It is not an average. Other life stages pass through unchanged. Both source stages, their score range, and their review flags remain in the review table.

In [4]:
def any_yes(values):
    # Replace missing values with blanks and convert all values to text.
    cleaned_values = values.fillna("").astype(str)

    # Ignore surrounding spaces and capitalization when checking for "yes".
    is_yes = cleaned_values.str.strip().str.lower().eq("yes")

    # Return one flag for the full group.
    return "Yes" if is_yes.any() else "No"


# Group by the keys used in scoring. Adult and juvenile migration share the
# combined "Migration" life_stage value and therefore enter the same group.
vulnerability_collapsed = (
    vulnerability.groupby(
        ["species", "life_stage", "limiting_factor"], as_index=False
    )
    .agg(
        # Retain source rank and recalculated-score ranges for review.
        vulnerability_rank_min=("vulnerability_rank", "min"),
        vulnerability_rank_max=("vulnerability_rank", "max"),
        vulnerability_score_min=("vulnerability_score", "min"),
        vulnerability_score_max=("vulnerability_score", "max"),

        # Use the larger adult/juvenile score in the scoring equations.
        vulnerability_score=("vulnerability_score", "max"),

        # Document how many and which source stages contributed.
        source_vulnerability_rows=("source_life_stage", "size"),
        source_life_stages=(
            "source_life_stage",
            lambda values: " | ".join(sorted(set(values.astype(str)))),
        ),

        # Combine review flags and uncertainty notes for the group.
        vulnerability_review_flag=("review_flag", any_yes),
        uncertainty_notes=(
            "uncertainty_flag",
            lambda values: " | ".join(
                sorted(set(values.dropna().astype(str)))
            ),
        ),
    )
)

display(vulnerability_collapsed.head(10))

,species,life_stage,limiting_factor,vulnerability_rank_min,vulnerability_rank_max,vulnerability_score_min,vulnerability_score_max,vulnerability_score,source_vulnerability_rows,source_life_stages,vulnerability_review_flag,uncertainty_notes
0,Bull Trout,FMO,Altered Flow Timing,11,11,0.292857,0.292857,0.292857,1,FMO (Fluvial),Yes,Unknown Future Climate Change Impacts
1,Bull Trout,FMO,Altered Primary Productivity,12,12,0.222143,0.222143,0.222143,1,FMO (Fluvial),No,
2,Bull Trout,FMO,Anthropogenic Barriers,3,3,0.858571,0.858571,0.858571,1,FMO (Fluvial),No,
3,Bull Trout,FMO,Channel and Habitat Structure,4,4,0.787857,0.787857,0.787857,1,FMO (Fluvial),No,
4,Bull Trout,FMO,Decreased Sediment Quantity,13,13,0.151429,0.151429,0.151429,1,FMO (Fluvial),Yes,Don't know how sediment quantity impacts FMO
5,Bull Trout,FMO,Decreased Water Quantity,2,2,0.929286,0.929286,0.929286,1,FMO (Fluvial),Yes,Unknown Future Climate Change Impacts
6,Bull Trout,FMO,Floodplain Condition,9,9,0.434286,0.434286,0.434286,1,FMO (Fluvial),No,
7,Bull Trout,FMO,Increased Sediment Quantity,14,14,0.080714,0.080714,0.080714,1,FMO (Fluvial),No,
8,Bull Trout,FMO,Non-Native Species Interactions and Competition,10,10,0.363571,0.363571,0.363571,1,FMO (Fluvial),No,
9,Bull Trout,FMO,Predation,7,7,0.575714,0.575714,0.575714,1,FMO (Fluvial),Yes,We don't know the extent of predation. Would b...


## 5. Level 1 calculations

Every summary is a different grouping of the same pathway contributions.

| Summary | Sum across | Question it supports |
|---|---|---|
| Species/life-stage risk | All 15 limiting factors | Which species and life-stage combination contributes most to this BSR's risk? |
| Limiting-factor risk | All species and life stages | Which limiting factor contributes most to this BSR's risk? |
| Overall BSR risk | All species, life stages, and limiting factors | What is the combined risk index for this BSR? |

The principal summaries are:

$$
\text{life-stage risk}
=
\text{life-stage fish use}
\times
\text{population priority}
\times
\sum_{\substack{\text{15 limiting}\\\text{factors}}}
\left(
\text{limiting-factor condition}
\times
\text{vulnerability}
\right)
$$

$$
\text{limiting-factor risk}
=
\text{limiting-factor condition}
\times
\sum_{\substack{\text{all species}\\\text{and life stages}}}
\left(
\text{life-stage fish use}
\times
\text{population priority}
\times
\text{vulnerability}
\right)
$$

$$
\text{overall BSR risk}
=
\sum_{\substack{\text{all species}\\\text{and life stages}}}
\text{life stage risk}
=
\sum_{\substack{\text{15 limiting}\\\text{factors}}}
\text{limiting factor risk}
$$

Summing the species and life-stage results, the species results, or the limiting-factor results must produce the same overall BSR risk.

Aggregate scores are sums and can exceed 1. Species and overall fish-use context fields do not enter these sums as additional multipliers.

The `highest_risk_*` fields identify the largest contributions, not a complete restoration-priority decision. Equal scores share a dense rank, and every tied top label is retained. If every contribution is zero, the labels describe a zero-score tie rather than evidence of a high-priority condition.

In [5]:
# Form one row for each BSR × species × life stage × limiting factor.
# The joins retain the source and review fields used in exported tables.
calculation_grid = (
    fish_use.merge(
        population[["basin", "species", "life_stage", "population_priority"]],
        on=["basin", "species", "life_stage"],
        how="left",
        validate="many_to_one",
    )
    .merge(
        vulnerability_collapsed,
        on=["species", "life_stage"],
        how="left",
        validate="many_to_many",
    )
    .merge(
        condition[
            [
                "bsr", "limiting_factor", "condition_score_raw_1_5",
                "condition_score_source", "condition_score", "n", "any_flag",
            ]
        ],
        on=["bsr", "limiting_factor"],
        how="left",
        validate="many_to_one",
    )
)

# Each risk contribution is the four-factor product in the Level 1 equation.
# Grouping these contributions in either direction gives the markdown totals.
calculation_grid["risk_component"] = (
    calculation_grid["LS_corrected_score"]
    * calculation_grid["population_priority"]
    * calculation_grid["condition_score"]
    * calculation_grid["vulnerability_score"]
)

display(
    calculation_grid[
        [
            "bsr", "species", "life_stage", "limiting_factor",
            "LS_corrected_score_source", "LS_corrected_score",
            "species_aggregate_score", "fish_use_score",
            "population_priority", "condition_score_raw_1_5",
            "condition_score_source", "condition_score",
            "vulnerability_rank_min", "vulnerability_rank_max",
            "vulnerability_score", "risk_component",
        ]
    ].head(10)
)


# Sum pathway risk across the 15 factors for each species and life stage:
# fish use × population priority × Σ(condition × vulnerability).
life_stage_scores = (
    calculation_grid.groupby(
        ["bsr", "basin", "species", "life_stage"], as_index=False
    )
    .agg(
        LS_corrected_score_source=("LS_corrected_score_source", "first"),
        LS_corrected_score=("LS_corrected_score", "first"),
        life_stage_fish_use_normalization_max=("life_stage_fish_use_normalization_max", "first"),
        species_aggregate_score=("species_aggregate_score", "first"),
        fish_use_score=("fish_use_score", "first"),
        population_priority=("population_priority", "first"),
        risk_score=("risk_component", "sum"),
        vulnerability_review_flag=("vulnerability_review_flag", any_yes),
    )
)
life_stage_scores["risk_rank_within_bsr"] = (
    life_stage_scores.groupby("bsr")["risk_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)
life_stage_scores["species_life_stage_label"] = (
    life_stage_scores["species"] + " | " + life_stage_scores["life_stage"]
)

# Species risk is the sum of its life-stage risks. Species fish use is context.
species_scores = (
    life_stage_scores.groupby(["bsr", "basin", "species"], as_index=False)
    .agg(
        species_aggregate_score=("species_aggregate_score", "first"),
        fish_use_score=("fish_use_score", "first"),
        life_stage_count=("life_stage", "size"),
        risk_score=("risk_score", "sum"),
        vulnerability_review_flag=("vulnerability_review_flag", any_yes),
    )
)
species_scores["risk_rank_within_bsr"] = (
    species_scores.groupby("bsr")["risk_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

# Sum pathway risk across all species and life stages for each factor:
# condition × Σ(fish use × population priority × vulnerability).
limiting_factor_scores = (
    calculation_grid.groupby(
        ["bsr", "basin", "limiting_factor"], as_index=False
    )
    .agg(
        fish_use_score=("fish_use_score", "first"),
        condition_score_raw_1_5=("condition_score_raw_1_5", "first"),
        condition_score_source=("condition_score_source", "first"),
        condition_score=("condition_score", "first"),
        condition_rating_n=("n", "first"),
        condition_review_flag=("any_flag", "first"),
        vulnerability_review_flag=("vulnerability_review_flag", any_yes),
        risk_score=("risk_component", "sum"),
    )
)
limiting_factor_scores["risk_rank_within_bsr"] = (
    limiting_factor_scores.groupby("bsr")["risk_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

source_fish_use = (
    fish_use.groupby(["bsr", "basin"], as_index=False)
    .agg(
        fish_use_score=("fish_use_score", "first"),
        fish_use_score_variants=("fish_use_score", "nunique"),
    )
)

# The BSR total is the sum of all life-stage risks. Independently summing
# limiting-factor and species risks is checked before export.
bsr_scores = (
    life_stage_scores.groupby(["bsr", "basin"], as_index=False)
    .agg(overall_risk_score=("risk_score", "sum"))
)

top_life_stage = (
    life_stage_scores.loc[life_stage_scores["risk_rank_within_bsr"].eq(1)]
    .groupby("bsr", as_index=False)
    .agg(
        highest_risk_species_life_stage=(
            "species_life_stage_label", lambda values: "; ".join(sorted(values))
        ),
        top_species_life_stage_risk_score=("risk_score", "first"),
        top_species_life_stage_risk_tie_count=("species_life_stage_label", "size"),
    )
)
top_limiting_factor = (
    limiting_factor_scores.loc[limiting_factor_scores["risk_rank_within_bsr"].eq(1)]
    .groupby("bsr", as_index=False)
    .agg(
        highest_risk_limiting_factor=(
            "limiting_factor", lambda values: "; ".join(sorted(values))
        ),
        top_limiting_factor_risk_score=("risk_score", "first"),
        top_limiting_factor_risk_tie_count=("limiting_factor", "size"),
    )
)
bsr_scores = (
    bsr_scores
    .merge(source_fish_use, on=["bsr", "basin"], validate="one_to_one")
    .merge(top_life_stage, on="bsr", validate="one_to_one")
    .merge(top_limiting_factor, on="bsr", validate="one_to_one")
)

display(
    bsr_scores[
        ["bsr", "overall_risk_score", "highest_risk_species_life_stage",
         "highest_risk_limiting_factor", "fish_use_score"]
    ].sort_values("overall_risk_score", ascending=False).head(10)
)

bsr_scores = bsr_scores.merge(
    identifier_review, on=["bsr", "basin"], how="left", validate="one_to_one"
)
bsr_scores["life_stage_fish_use_normalization_max"] = life_stage_fish_use_source_max
bsr_scores["scoring_run_id"] = RUN_ID
bsr_scores["scoring_framework_version"] = FRAMEWORK_VERSION


,bsr,species,life_stage,limiting_factor,LS_corrected_score_source,LS_corrected_score,species_aggregate_score,fish_use_score,population_priority,condition_score_raw_1_5,condition_score_source,condition_score,vulnerability_rank_min,vulnerability_rank_max,vulnerability_score,risk_component
0,CC1,Chinook,Spawning,Altered Flow Timing,0.0,0.0,0.77,0.4162,0.12,4.31,0.84475,0.829225,10,10,0.363571,0.0
1,CC1,Chinook,Spawning,Altered Primary Productivity,0.0,0.0,0.77,0.4162,0.12,4.31,0.84475,0.829225,14,14,0.080714,0.0
2,CC1,Chinook,Spawning,Anthropogenic Barriers,0.0,0.0,0.77,0.4162,0.12,3.68,0.70300,0.673300,15,15,0.010000,0.0
3,CC1,Chinook,Spawning,Channel and Habitat Structure,0.0,0.0,0.77,0.4162,0.12,4.57,0.90325,0.893575,5,5,0.717143,0.0
4,CC1,Chinook,Spawning,Decreased Sediment Quantity,0.0,0.0,0.77,0.4162,0.12,2.95,0.53875,0.492625,3,3,0.858571,0.0
5,CC1,Chinook,Spawning,Decreased Water Quantity,0.0,0.0,0.77,0.4162,0.12,5.00,1.00000,1.000000,2,2,0.929286,0.0
6,CC1,Chinook,Spawning,Floodplain Condition,0.0,0.0,0.77,0.4162,0.12,4.78,0.95050,0.945550,6,6,0.646429,0.0
7,CC1,Chinook,Spawning,Increased Sediment Quantity,0.0,0.0,0.77,0.4162,0.12,4.51,0.88975,0.878725,11,11,0.292857,0.0
8,CC1,Chinook,Spawning,Non-Native Species Interactions and Competition,0.0,0.0,0.77,0.4162,0.12,4.64,0.91900,0.910900,12,12,0.222143,0.0
9,CC1,Chinook,Spawning,Predation,0.0,0.0,0.77,0.4162,0.12,4.22,0.82450,0.806950,9,9,0.434286,0.0


,bsr,overall_risk_score,highest_risk_species_life_stage,highest_risk_limiting_factor,fish_use_score
24,UGR5,5.141376,Chinook | Migration,Decreased Water Quantity,0.5242
22,UGR3,4.360243,Chinook | Migration,Predation,0.6514
0,CC1,3.779498,Chinook | Migration,Decreased Water Quantity,0.4162
26,UGR7,3.233427,Chinook | Migration,Channel and Habitat Structure,0.3375
5,CC6,2.791313,Bull Trout | FMO,Decreased Water Quantity,0.6329
17,UGR17,2.789539,Bull Trout | FMO,Channel and Habitat Structure,0.6039
4,CC5,2.597345,Bull Trout | FMO,Channel and Habitat Structure,0.4663
11,UGR11,2.532195,Chinook | Migration,Channel and Habitat Structure,0.4504
6,CC7,2.463117,Bull Trout | Spawning & Resident,Summer Water Temperature,1.0000
21,UGR20,2.449949,Bull Trout | Spawning & Resident,Non-Native Species Interactions and Competition,0.8697


## 7. Calculate Level 2 action alignment

For each action and limiting factor, the source action relationship weight is:

$$\text{action relationship weight} = \text{directness} \times \text{frequency}$$

The notebook applies that weight to limiting-factor risk.

| Output field | Calculation for one BSR and action | What it includes |
|---|---|---|
| `action_benefit_score` | $\sum_{\text{limiting factors}} (\text{limiting-factor risk} \times \text{action relationship weight})$ | Also includes population priority |

These indices do **not** estimate the amount of condition improvement, habitat gain, or fish response. They do not account for cost, feasibility, implementation constraints, landowner willingness, or site-specific effectiveness. Adding a species or an action can change aggregate totals.

### What the overall action total means

$$\text{overall benefit score} = \sum_{\text{actions}} \text{action benefit score}$$

A limiting factor receives more effective weight if it is associated with more actions or stronger action relationships. The overall benefit score can therefore rank BSRs differently from overall risk, and it is not an additive estimate of realized benefits. The table displayed below reports each factor's summed action weight so this effect is visible.

In [6]:
# Each BSR × factor × action row contributes factor risk × action weight.
action_components = limiting_factor_scores.merge(
    lfat[
        [
            "action_id", "action_type", "action_definition",
            "limiting_factor", "directness_code", "directness_value",
            "frequency_code", "frequency_value", "lfat_score",
        ]
    ],
    on="limiting_factor",
    how="inner",
    validate="many_to_many",
)
action_components["benefit_component"] = (
    action_components["risk_score"] * action_components["lfat_score"]
)

# Action benefit is the sum of these weighted factor risks within each BSR.
action_scores = (
    action_components.groupby(
        ["bsr", "basin", "action_id", "action_type", "action_definition"],
        as_index=False,
    )
    .agg(
        action_benefit_score=("benefit_component", "sum"),
        condition_review_count=(
            "condition_review_flag",
            lambda values: int(
                values.fillna(False)
                .astype(str)
                .str.strip()
                .str.lower()
                .isin(["true", "yes", "1"])
                .sum()
            ),
        ),
        vulnerability_review_count=(
            "vulnerability_review_flag",
            lambda values: int(values.eq("Yes").sum()),
        ),
    )
)
action_scores["benefit_score_label"] = (
    action_scores["action_type"].astype(str) + " action alignment score"
)
action_scores["benefit_rank_within_bsr"] = (
    action_scores.groupby("bsr")["action_benefit_score"]
    .rank(method="dense", ascending=False)
    .astype(int)
)

top_action = (
    action_scores.loc[action_scores["benefit_rank_within_bsr"].eq(1)]
    .assign(
        priority_action_label=lambda table: (
            table["action_id"].astype(str) + " | " + table["action_type"]
        )
    )
    .groupby("bsr", as_index=False)
    .agg(
        highest_risk_aligned_action_type=(
            "priority_action_label", lambda values: "; ".join(sorted(values))
        ),
        highest_action_benefit_score=("action_benefit_score", "first"),
        top_action_benefit_tie_count=("priority_action_label", "size"),
    )
)

# Overall benefit is the sum of all action benefit scores in the BSR.
overall_action_scores = (
    action_scores.groupby("bsr", as_index=False)
    .agg(overall_benefit_score=("action_benefit_score", "sum"))
)
bsr_scores = (
    bsr_scores
    .merge(top_action, on="bsr", validate="one_to_one")
    .merge(overall_action_scores, on="bsr", validate="one_to_one")
)

action_factor_weights = lfat.groupby("limiting_factor", as_index=False).agg(
    summed_action_weight=("lfat_score", "sum"),
    positive_weight_actions=("lfat_score", lambda values: int(values.gt(0).sum())),
    listed_actions=("action_id", "nunique"),
).sort_values("summed_action_weight", ascending=False).reset_index(drop=True)

display(action_factor_weights)
display(
    action_scores[
        ["bsr", "action_type", "action_benefit_score", "benefit_rank_within_bsr"]
    ].sort_values(["bsr", "benefit_rank_within_bsr"]).head(15)
)
display(
    bsr_scores[
        ["bsr", "overall_risk_score", "overall_benefit_score", "input_review_status"]
    ].sort_values("overall_risk_score", ascending=False).head(10)
)


,limiting_factor,summed_action_weight,positive_weight_actions,listed_actions
0,Summer Water Temperature,3.63,11,11
1,Floodplain Condition,3.62,11,11
2,Side Channel and Wetland Habitat,3.59,11,11
3,Riparian Condition,3.08,11,11
4,Altered Primary Productivity,3.06,11,11
5,Water Quality,2.79,11,11
6,Channel and Habitat Structure,2.56,11,11
7,Increased Sediment Quantity,2.54,11,11
8,Winter Water Temperature,1.91,11,11
9,Decreased Water Quantity,1.86,11,11


,bsr,action_type,action_benefit_score,benefit_rank_within_bsr
2,CC1,Floodplain – Reconnect and Restore,1.635210,1
3,CC1,Riparian Vegetation Restoration,1.429006,2
5,CC1,Instream Flow Restoration,1.318334,3
1,CC1,Instream Complexity Improvement,1.242500,4
0,CC1,Protect Land (Easement and Acquisition),0.835179,5
4,CC1,Fish Passage – Barrier Removal and Replacement,0.643076,6
7,CC1,Water Quality Improvement (not including tempe...,0.555634,7
6,CC1,"Thermal Refuge Enhancement (reconnect, expand)",0.548629,8
10,CC1,Species Management (non-native or unnatural),0.511584,9
9,CC1,Upland Treatments,0.430125,10


,bsr,overall_risk_score,overall_benefit_score,input_review_status
24,UGR5,5.141376,12.723637,Exact BSR identifier
22,UGR3,4.360243,10.108133,Exact BSR identifier
0,CC1,3.779498,9.397497,Provisional: BSR correspondence requires review
26,UGR7,3.233427,8.180237,Exact BSR identifier
5,CC6,2.791313,7.178508,Provisional: BSR correspondence requires review
17,UGR17,2.789539,7.645297,Exact BSR identifier
4,CC5,2.597345,6.613427,Provisional: BSR correspondence requires review
11,UGR11,2.532195,6.573266,Exact BSR identifier
6,CC7,2.463117,6.401734,Provisional: BSR correspondence requires review
21,UGR20,2.449949,5.801986,Exact BSR identifier


## 8. Validate the completed calculations

The checks independently sum pathway components and compare them with the life-stage, species, limiting-factor, and BSR summaries. They also verify every action total, expected record coverage, zero-fish-use behavior, and finite scores.

A passing result means the tables and equations agree. It does not establish that provisional BSR matches, condition-rating assumptions, vulnerability ranks, or action weights are biologically correct. Those source review items remain separately identified.

In [7]:
# All checks in this cell run before any output file is staged or published.
calculation_qc_passed = False
worked_example_qc_passed = False
stage = "calculation"
check(len(vulnerability_collapsed) == len(stage_keys) * 15, "Collapsed vulnerability row count", stage=stage)
check(not vulnerability_collapsed.duplicated(["species", "life_stage", "limiting_factor"]).any(), "Unique collapsed vulnerability keys", stage=stage)
check(len(calculation_grid) == len(fish_use) * 15, "Complete pathway-grid row count", stage=stage)
check(not calculation_grid.duplicated(["bsr", "species", "life_stage", "limiting_factor"]).any(), "Unique pathway keys", stage=stage)
check(calculation_grid.groupby("bsr").size().eq(len(stage_keys) * 15).all(), "Complete pathway coverage in each BSR", stage=stage)
for column in ["LS_corrected_score", "species_aggregate_score", "fish_use_score", "population_priority", "condition_score", "vulnerability_score", "risk_component"]:
    check(np.isfinite(calculation_grid[column]).all(), f"Finite joined {column}", stage=stage)
check(
    np.allclose(
        calculation_grid["risk_component"],
        calculation_grid["LS_corrected_score"]
        * calculation_grid["population_priority"]
        * calculation_grid["condition_score"]
        * calculation_grid["vulnerability_score"],
        rtol=1e-12, atol=1e-12,
    ),
    "Four-factor pathway risk equation", stage=stage,
)
zero = calculation_grid["LS_corrected_score"].eq(0)
check(calculation_grid.loc[zero, "risk_component"].eq(0).all(), "Zero fish use gives zero pathway risk", stage=stage)

# Verify each Level 1 grouping directly against the pathway products.
for table_name, table, keys in [
    ("life stage", life_stage_scores, ["bsr", "basin", "species", "life_stage"]),
    ("species", species_scores, ["bsr", "basin", "species"]),
    ("limiting factor", limiting_factor_scores, ["bsr", "basin", "limiting_factor"]),
]:
    expected = calculation_grid.groupby(keys, as_index=False).agg(
        risk_score=("risk_component", "sum")
    ).sort_values(keys)
    actual = table[keys + ["risk_score"]].sort_values(keys)
    compare_frames(actual, expected, f"{table_name} risk equals pathway sum")

keys = ["bsr", "basin", "species", "life_stage"]
detail_fields = ["LS_corrected_score_source", "LS_corrected_score", "species_aggregate_score", "life_stage_fish_use_normalization_max"]
compare_frames(life_stage_scores[keys + detail_fields].sort_values(keys), fish_use[keys + detail_fields].sort_values(keys), "Source and normalized fish-use fields preserved")
for table_name, table in [
    ("life stage", life_stage_scores),
    ("species", species_scores),
    ("limiting factor", limiting_factor_scores),
]:
    total = table.groupby("bsr")["risk_score"].sum().sort_index()
    check(
        np.allclose(total, bsr_scores.set_index("bsr")["overall_risk_score"].sort_index(), rtol=0, atol=1e-12),
        f"Overall BSR risk equals {table_name} risk sum", stage=stage,
    )
check(set(bsr_scores["bsr"]) == set(fish_use["bsr"]), "BSR summary coverage", stage=stage)
check(not bsr_scores["bsr"].duplicated().any(), "One summary row per BSR", stage=stage)
check(len(action_components) == len(bsr_scores) * len(lfat), "Complete action-component coverage", stage=stage)
check(len(action_scores) == len(bsr_scores) * lfat["action_id"].nunique(), "Complete BSR/action coverage", stage=stage)
check(not action_scores.duplicated(["bsr", "action_id"]).any(), "Unique BSR/action keys", stage=stage)

check(
    np.allclose(action_components["benefit_component"], action_components["risk_score"] * action_components["lfat_score"], rtol=1e-12, atol=1e-12),
    "Action benefit component equals factor risk × action weight", stage=stage,
)
action_keys = ["bsr", "action_id"]
expected_actions = action_components.groupby(action_keys, as_index=False).agg(
    action_benefit_score=("benefit_component", "sum")
)
compare_frames(
    action_scores[action_keys + ["action_benefit_score"]].sort_values(action_keys),
    expected_actions.sort_values(action_keys),
    "Action benefit equals weighted factor risk sum",
)
expected_overall = action_scores.groupby("bsr", as_index=False).agg(
    overall_benefit_score=("action_benefit_score", "sum")
)
compare_frames(
    bsr_scores[expected_overall.columns].sort_values("bsr"),
    expected_overall.sort_values("bsr"),
    "Overall benefit equals the sum across actions",
)

# A second calculation applies each factor's total action weight just once.
effective = limiting_factor_scores.merge(
    action_factor_weights[["limiting_factor", "summed_action_weight"]],
    on="limiting_factor", validate="many_to_one",
)
effective["contribution"] = effective["risk_score"] * effective["summed_action_weight"]
effective_total = effective.groupby("bsr")["contribution"].sum().sort_index()
check(
    np.allclose(effective_total, bsr_scores.set_index("bsr")["overall_benefit_score"].sort_index(), rtol=1e-12, atol=1e-12),
    "Overall action total equals factor risk × summed action weight", stage=stage,
)
for name, table in [("life stage", life_stage_scores), ("species", species_scores), ("limiting factor", limiting_factor_scores), ("action", action_scores), ("BSR", bsr_scores)]:
    scores = [column for column in table if column.endswith("_score")]
    check(np.isfinite(table[scores].to_numpy(dtype=float)).all(), f"{name}: finite output scores", stage=stage)

def scoring_state_signature():
    """Detect scoring-table edits made after validation in a live kernel."""
    digest = hashlib.sha256()
    for name in ["fish_use", "population", "condition", "vulnerability", "lfat", "vulnerability_collapsed", "calculation_grid", "life_stage_scores", "species_scores", "limiting_factor_scores", "action_components", "action_scores", "bsr_scores", "identifier_review"]:
        table = globals()[name]
        digest.update(name.encode("utf-8"))
        digest.update(repr(table.columns.tolist()).encode("utf-8"))
        digest.update(pd.util.hash_pandas_object(table, index=True).to_numpy().tobytes())
    digest.update(repr(float(life_stage_fish_use_source_max)).encode("utf-8"))
    return digest.hexdigest()


validated_calculation_signature = scoring_state_signature()
calculation_qc_passed = True
print(f"Input, transformation, and calculation checks passed: {len(qc_records)}.")
print(f"Validated {len(bsr_scores)} BSRs, {len(calculation_grid):,} pathways, and {lfat['action_id'].nunique()} action types.")
print("Input review flags remain separate from this numerical result.")


Input, transformation, and calculation checks passed: 166.
Validated 29 BSRs, 4,350 pathways, and 11 action types.
Input review flags remain separate from this numerical result.


## 9. A small example to check the logic

Consider two life stages, two limiting factors, and two actions. Spawning has zero fish use, so its contributions must all be zero. Rearing has fish use 0.8 and population priority 0.4. For rearing, condition × vulnerability totals $0.10\times0.50+1.00\times1.00=1.05$.

Therefore **rearing risk = 0.8 × 0.4 × 1.05 = 0.336**. The limiting-factor risks are 0.016 and 0.320, which sum to the same total. Applying the two action-weight sets gives action-alignment scores 0.328 and 0.176; their overall alignment total is 0.504. The larger overall action total reflects repeated action relationships, not additional independently realized benefit.

The example below uses explicit expected results and runs before export. It illustrates the equations with simplified values, not an actual BSR.

In [8]:
worked_example_qc_passed = False

test_population = {"Spawning": 0.60, "Rearing": 0.40}
test_life_stage_fish_use = {"Spawning": 0.00, "Rearing": 0.80}
test_condition = {"Temperature": 0.10, "Instream Complexity": 1.00}
test_vulnerability = {
    ("Spawning", "Temperature"): 1.00,
    ("Rearing", "Temperature"): 0.50,
    ("Spawning", "Instream Complexity"): 0.50,
    ("Rearing", "Instream Complexity"): 1.00,
}
test_action_weight = {
    ("Floodplain Restoration", "Temperature"): 0.50,
    ("Floodplain Restoration", "Instream Complexity"): 1.00,
    ("Riparian Planting", "Temperature"): 1.00,
    ("Riparian Planting", "Instream Complexity"): 0.50,
}

# Calculate one four-factor risk contribution for each pathway.
test_grid = pd.DataFrame([
    {
        "species": "Test Salmon",
        "life_stage": life_stage,
        "limiting_factor": limiting_factor,
        "fish_use": test_life_stage_fish_use[life_stage],
        "risk_component": (
            test_life_stage_fish_use[life_stage]
            * test_population[life_stage]
            * condition_score
            * test_vulnerability[(life_stage, limiting_factor)]
        ),
    }
    for life_stage in test_population
    for limiting_factor, condition_score in test_condition.items()
])

test_life = test_grid.groupby("life_stage", as_index=False).agg(
    risk_score=("risk_component", "sum")
)
test_species = test_grid.groupby("species", as_index=False).agg(
    risk_score=("risk_component", "sum")
)
test_lf = test_grid.groupby("limiting_factor", as_index=False).agg(
    risk_score=("risk_component", "sum")
)

test_actions = pd.DataFrame([
    {
        "action": action,
        "action_benefit_score": sum(
            row.risk_score * test_action_weight[(action, row.limiting_factor)]
            for row in test_lf.itertuples(index=False)
        ),
    }
    for action in ["Floodplain Restoration", "Riparian Planting"]
])
test_overall_benefit = test_actions["action_benefit_score"].sum()

# These are independently calculated from the values in the markdown.
assert np.isclose(test_life.set_index("life_stage").loc["Spawning", "risk_score"], 0)
assert np.isclose(test_life.set_index("life_stage").loc["Rearing", "risk_score"], 0.336)
assert test_grid.loc[test_grid["fish_use"].eq(0), "risk_component"].eq(0).all()
assert np.isclose(test_species["risk_score"].sum(), 0.336)
assert np.allclose(
    test_lf.set_index("limiting_factor").loc[
        ["Temperature", "Instream Complexity"], "risk_score"
    ], [0.016, 0.320]
)
assert np.isclose(test_life["risk_score"].sum(), test_lf["risk_score"].sum())
assert np.allclose(test_actions["action_benefit_score"], [0.328, 0.176])
assert np.isclose(test_overall_benefit, 0.504)

worked_example_qc_passed = True
print("Worked example passed. Output-file checks have not yet run.")
check(True, "Hand-calculated example", stage="calculation")
display(test_life)
display(test_species)
display(test_lf)
display(test_actions)
print(f"Overall benefit score: {test_overall_benefit:g}")


Worked example passed. Output-file checks have not yet run.


,life_stage,risk_score
0,Rearing,0.336
1,Spawning,0.000


,species,risk_score
0,Test Salmon,0.336


,limiting_factor,risk_score
0,Instream Complexity,0.320
1,Temperature,0.016


,action,action_benefit_score
0,Floodplain Restoration,0.328
1,Riparian Planting,0.176


Overall benefit score: 0.504


## 10. Prepare export tables and their documentation

The eight existing core CSV filenames and `bsr_scores.gpkg` are retained. Level 2 scores use the risk-weighted `action_benefit_score` and its `overall_benefit_score` sum. BSR outputs also retain the source crosswalk status, automatic provisional-match flag, fish-use normalization denominator, and run and framework identifiers.

The companion `framework.md` documents the implemented scoring framework and output-field meanings. `QC` contains pathway support tables, BSR and vulnerability source-review information, normalization details, population-priority sums, effective action weights, field definitions, input-file hashes, and the QC record for the run.

The scored GeoPackage preserves the input geometry and spatial index and includes nonspatial fish-use, population, life-stage, species, limiting-factor, and action tables for joins. The spatial output must contain exactly the scored BSRs.

In [9]:
CORE_SCORE_FILES = {
    "bsr": "bsr_scores.csv",
    "fish_use": "fish_use_scores.csv",
    "population": "population_scores.csv",
    "life_stage": "life_stage_scores.csv",
    "species": "species_scores.csv",
    "limiting_factor": "limiting_factor_scores_integrated.csv",
    "action": "action_scores.csv",
    "grid": "calculation_grid.csv",
}

CORE_OUTPUTS = {
    CORE_SCORE_FILES["bsr"]: bsr_scores,
    CORE_SCORE_FILES["fish_use"]: fish_use,
    CORE_SCORE_FILES["population"]: population,
    CORE_SCORE_FILES["life_stage"]: life_stage_scores,
    CORE_SCORE_FILES["species"]: species_scores,
    CORE_SCORE_FILES["limiting_factor"]: limiting_factor_scores,
    CORE_SCORE_FILES["action"]: action_scores,
    CORE_SCORE_FILES["grid"]: calculation_grid,
}
QC_OUTPUTS = {
    "action_score_components.csv": action_components,
    "assumptions_for_review.csv": assumptions,
    "bsr_identifiers_for_review.csv": identifier_review,
    "vulnerability_scores_for_review.csv": vulnerability_collapsed,
}

GPKG_ATTRIBUTE_TABLES = {
    "fish_use_scores": {
        "table": fish_use,
        "identifier": "Atlas fish-use scores",
        "description": (
            "Source and normalized overall, species, and life-stage fish-use scores"
        ),
    },
    "population_scores": {
        "table": population,
        "identifier": "Atlas population scores",
        "description": (
            "Source population-priority scores by basin, species, and life stage"
        ),
    },
    "life_stage_scores": {
        "table": life_stage_scores,
        "identifier": "Atlas life-stage scores",
        "description": (
            "Level 1 species and life-stage scores by BSR for attribute joins"
        ),
    },
    "species_scores": {
        "table": species_scores,
        "identifier": "Atlas species scores",
        "description": (
            "Level 1 species risk scores by BSR"
        ),
    },
    "limiting_factor_scores": {
        "table": limiting_factor_scores,
        "identifier": "Atlas limiting-factor scores",
        "description": (
            "Level 1 limiting-factor scores by BSR for attribute joins"
        ),
    },
    "action_type_scores": {
        "table": action_scores,
        "identifier": "Atlas action-type scores",
        "description": (
            "Level 2 action-alignment indices by BSR"
        ),
    },
}


# A concise field dictionary defines the established output schema.
field_definitions = pd.DataFrame([
    ["fish_use / life_stage / grid", "LS_corrected_score_source", "Source Life-Stage Fish Use", "Unchanged input value before maximum normalization."],
    ["fish_use / life_stage / grid", "LS_corrected_score", "Life-Stage Fish Use Score", "Source divided by the recorded global maximum; 0–1; risk multiplier."],
    ["fish_use / life_stage / bsr / grid", "life_stage_fish_use_normalization_max", "Life-Stage Fish-Use Normalization Maximum", "Denominator used for this run; compare before interpreting changes between runs."],
    ["fish_use / life_stage / species", "species_aggregate_score", "Species Fish Use Score", "Unchanged context field; may exceed 1; not a multiplier."],
    ["fish_use / bsr", "fish_use_score", "Overall Fish Use Score", "Input fish_use_score_decimal, unchanged; 0–1; context only."],
    ["limiting_factor / grid", "condition_score", "Limiting-Factor Impairment Score", "Raw 1–5 mapped to 0.01–1; assumes higher raw values indicate greater impairment."],
    ["population / life_stage / grid", "population_priority", "Life-Stage Population Priority", "Within basin/species weight; source rounding retained; no explicit between-species multiplier."],
    ["bsr", "overall_risk_score", "Overall Risk Score", "Sum of population-weighted pathway contributions; relative index, not a probability."],
    ["action", "action_benefit_score", "Action Benefit Score", "Sum of limiting-factor risk × action relationship weight for one action; alignment index."],
    ["bsr", "overall_benefit_score", "Overall Benefit Score", "Sum across actions; repeated limiting-factor relationships; not additive realized benefits."],
    ["bsr", "highest_action_benefit_score", "Highest Action Benefit Score", "Largest action-specific alignment score within a BSR."],
    ["bsr / fish_use / grid", "bsr_crosswalk_status", "Source BSR Crosswalk Status", "Original source status preserved, including provisional_positional."],
    ["bsr / fish_use / grid", "bsr_match_review_required", "BSR Correspondence Requires Review", "True when the source crosswalk status is not exact_identifier."],
], columns=["tables", "field", "display_label", "definition"])
run_metadata = {
    "run_id": RUN_ID,
    "created_utc": RUN_CREATED_UTC,
    "framework_version": FRAMEWORK_VERSION,
    "life_stage_fish_use_normalization": normalization_summary.iloc[0].to_dict(),
    "normalization_maximum_source_rows": maximum_rows.to_dict("records"),
    "condition_rating_assumption": "1 = least impairment; 5 = greatest impairment",
    "unresolved_bsr_matches": unresolved_bsrs,
    "population_priorities": "Source values retained; sums checked within basin/species with absolute tolerance 0.011.",
    "run_review_status": "provisional BSR correspondence" if unresolved_bsrs else "exact BSR identifiers",
    "source_vulnerability_review_rows": int(vulnerability["review_flag"].astype(str).str.strip().str.casefold().eq("yes").sum()),
    "source_condition_review_rows": int(condition["any_flag"].astype(str).str.strip().str.casefold().isin(["true", "yes", "1"]).sum()),
    "input_files": {key: {"file_name": path.name, "sha256": input_file_hashes[key]} for key, path in {**INPUT_PATHS, "spatial": BSR_INPUT_PATH}.items()},
    "core_output_rows": {name: len(table) for name, table in CORE_OUTPUTS.items()},
    "numerical_qc": "passed before staging",
}
QC_OUTPUTS.update({
    "input_file_summary.csv": input_summary,
    "life_stage_fish_use_normalization.csv": normalization_summary,
    "normalization_maximum_source_rows.csv": maximum_rows,
    "population_priority_sums.csv": population_review,
    "action_factor_weight_totals.csv": action_factor_weights,
    "score_field_definitions.csv": field_definitions,
})

### GeoPackage helper functions

These functions copy the source GeoPackage, add the score and review fields, and register the nonspatial score tables. They do not transform or rewrite polygon geometry. Existing spatial-index triggers are restored after attribute updates, and the next section checks geometry, feature IDs, index rows, and trigger definitions against the source.

The helper writes only to the temporary export location supplied by the final cell. A failure here leaves the published outputs in place.

In [10]:
def quote_identifier(name):
    return '"' + str(name).replace('"', '""') + '"'


def sqlite_type(series):
    if pd.api.types.is_bool_dtype(series) or pd.api.types.is_integer_dtype(series):
        return "INTEGER"
    if pd.api.types.is_numeric_dtype(series):
        return "REAL"
    return "TEXT"


def sqlite_value(value):
    if pd.isna(value):
        return None
    if isinstance(value, np.generic):
        return value.item()
    return value


def geometry_digest(path, feature_table, key_field, geometry_field):
    digest = hashlib.sha256()
    query = (
        f"SELECT {quote_identifier(key_field)}, "
        f"{quote_identifier(geometry_field)} "
        f"FROM {quote_identifier(feature_table)} "
        f"ORDER BY {quote_identifier(key_field)}"
    )
    with closing(sqlite3.connect(path)) as connection:
        for key, geometry in connection.execute(query):
            digest.update(str(key).encode("utf-8"))
            digest.update(bytes(geometry) if geometry is not None else b"")
    return digest.hexdigest()


def write_attribute_table(
    connection, table_name, table, identifier, description
):
    columns = table.columns.tolist()
    normalized_columns = [str(column).lower() for column in columns]
    if len(normalized_columns) != len(set(normalized_columns)):
        raise ValueError(
            f"GeoPackage table {table_name} has duplicate column names."
        )
    if "fid" in normalized_columns:
        raise ValueError(
            f"GeoPackage table {table_name} already contains a fid field."
        )
    table_exists = connection.execute(
        "SELECT 1 FROM sqlite_master WHERE name = ?", (table_name,)
    ).fetchone()
    if table_exists is not None:
        raise ValueError(
            f"GeoPackage already contains a table named {table_name}."
        )

    column_definitions = ["fid INTEGER PRIMARY KEY AUTOINCREMENT"]
    column_definitions.extend(
        f"{quote_identifier(column)} {sqlite_type(table[column])}"
        for column in columns
    )
    connection.execute(
        f"CREATE TABLE {quote_identifier(table_name)} "
        f"({', '.join(column_definitions)})"
    )

    placeholders = ", ".join("?" for _ in columns)
    insert_sql = (
        f"INSERT INTO {quote_identifier(table_name)} "
        f"({', '.join(quote_identifier(column) for column in columns)}) "
        f"VALUES ({placeholders})"
    )
    connection.executemany(
        insert_sql,
        (
            tuple(sqlite_value(value) for value in row)
            for row in table.itertuples(index=False, name=None)
        ),
    )
    connection.execute(
        "INSERT INTO gpkg_contents "
        "(table_name, data_type, identifier, description, last_change) "
        "VALUES (?, 'attributes', ?, ?, "
        "strftime('%Y-%m-%dT%H:%M:%fZ', 'now'))",
        (table_name, identifier, description),
    )
    if "bsr" in normalized_columns:
        bsr_column = columns[normalized_columns.index("bsr")]
        index_name = f"idx_{table_name}_bsr"
        connection.execute(
            f"CREATE INDEX {quote_identifier(index_name)} "
            f"ON {quote_identifier(table_name)} "
            f"({quote_identifier(bsr_column)})"
        )


def write_scored_bsr_gpkg(
    input_path, output_path, summary, attribute_tables
):
    # sqlite3 connections are closed explicitly because its context
    # manager commits or rolls back but does not close the file handle.
    # An open handle prevents Path.replace() on Windows.
    with closing(sqlite3.connect(input_path)) as connection:
        feature_tables = [
            row[0]
            for row in connection.execute(
                "SELECT table_name FROM gpkg_contents WHERE data_type = 'features'"
            )
        ]
        candidates = []
        for table_name in feature_tables:
            columns = [
                row[1]
                for row in connection.execute(
                    f"PRAGMA table_info({quote_identifier(table_name)})"
                )
            ]
            source_key = next(
                (column for column in columns if column.lower() == "bsr"),
                None,
            )
            if source_key is not None:
                candidates.append((table_name, source_key, columns))

        if len(candidates) != 1:
            raise ValueError(
                "Expected exactly one feature layer containing a BSR field; "
                f"found {len(candidates)}."
            )
        feature_table, source_key, source_columns = candidates[0]
        geometry_row = connection.execute(
            "SELECT column_name FROM gpkg_geometry_columns WHERE table_name = ?",
            (feature_table,),
        ).fetchone()
        if geometry_row is None:
            raise ValueError(
                f"No geometry field is registered for layer {feature_table}."
            )
        geometry_field = geometry_row[0]
        spatial_keys = {
            str(row[0]).strip()
            for row in connection.execute(
                f"SELECT {quote_identifier(source_key)} "
                f"FROM {quote_identifier(feature_table)}"
            )
        }

    summary_keys = set(summary["bsr"].astype(str).str.strip())
    if spatial_keys != summary_keys:
        raise ValueError(
            "The GeoPackage and score summary have different BSR coverage. "
            f"Missing scores: {sorted(spatial_keys - summary_keys)}; "
            f"missing geometry: {sorted(summary_keys - spatial_keys)}"
        )

    scored = summary.rename(columns={"bsr": "score_bsr"}).copy()
    scored_columns = scored.columns.tolist()
    existing_lower = {column.lower() for column in source_columns}
    collisions = [
        column for column in scored_columns if column.lower() in existing_lower
    ]
    if collisions:
        raise ValueError(
            "Score fields collide with existing GeoPackage fields: "
            + ", ".join(collisions)
        )

    temporary_path = output_path.with_name(
        f".{output_path.stem}.{uuid4().hex}.tmp.gpkg"
    )
    shutil.copy2(input_path, temporary_path)

    try:
        with closing(sqlite3.connect(temporary_path)) as connection:
            # Only attributes are updated. Suspend existing RTree triggers
            # during those updates, then restore their exact SQL. Geometry,
            # feature IDs, index rows, and trigger definitions are checked
            # against the source before this staged file can be published.
            prefix = f"rtree_{feature_table}_{geometry_field}_"
            spatial_triggers = [
                (name, sql) for name, sql in connection.execute(
                    "SELECT name, sql FROM sqlite_master WHERE type='trigger' AND tbl_name=?",
                    (feature_table,),
                ) if name.startswith(prefix)
            ]
            with connection:
                for trigger_name, _ in spatial_triggers:
                    connection.execute(f"DROP TRIGGER {quote_identifier(trigger_name)}")
                for column in scored_columns:
                    connection.execute(
                        f"ALTER TABLE {quote_identifier(feature_table)} "
                        f"ADD COLUMN {quote_identifier(column)} "
                        f"{sqlite_type(scored[column])}"
                    )

                assignments = ", ".join(
                    f"{quote_identifier(column)} = ?"
                    for column in scored_columns
                )
                update_sql = (
                    f"UPDATE {quote_identifier(feature_table)} "
                    f"SET {assignments} "
                    f"WHERE TRIM(CAST("
                    f"{quote_identifier(source_key)} AS TEXT)) = ?"
                )
                for original_bsr, (_, row) in zip(
                    summary["bsr"].astype(str), scored.iterrows()
                ):
                    values = [
                        sqlite_value(row[column])
                        for column in scored_columns
                    ]
                    values.append(original_bsr.strip())
                    cursor = connection.execute(update_sql, values)
                    if cursor.rowcount != 1:
                        raise ValueError(
                            f"Expected one spatial row for {original_bsr}; "
                            f"updated {cursor.rowcount}."
                        )

                connection.execute(
                    "UPDATE gpkg_contents "
                    "SET identifier = ?, description = ?, last_change = strftime('%Y-%m-%dT%H:%M:%fZ', 'now') "
                    "WHERE table_name = ?",
                    (
                        "Atlas scored BSRs",
                        "BSR geometry with Level 1 risk, Level 2 alignment, and input-review status",
                        feature_table,
                    ),
                )

                for table_name, table_specification in attribute_tables.items():
                    write_attribute_table(
                        connection=connection,
                        table_name=table_name,
                        **table_specification,
                    )

                for _, trigger_sql in spatial_triggers:
                    connection.execute(trigger_sql)

            integrity = connection.execute(
                "PRAGMA integrity_check"
            ).fetchone()[0]
            if integrity != "ok":
                raise ValueError(
                    f"GeoPackage integrity check failed: {integrity}"
                )

        # Path.replace() overwrites an existing output atomically. The SQLite
        # connection must be closed before this line on Windows.
        try:
            temporary_path.replace(output_path)
        except PermissionError as error:
            raise PermissionError(
                f"Could not replace {output_path}. Close the output "
                "GeoPackage in QGIS, ArcGIS, or another application, then "
                "run this cell again."
            ) from error
    finally:
        # Do not let cleanup of a staging file hide the original exception.
        try:
            temporary_path.unlink(missing_ok=True)
        except OSError:
            pass

    return (
        feature_table, geometry_field, source_key, scored_columns,
        list(attribute_tables),
    )

## 11. Verify temporary files, then publish the complete output set

This final cell stages the new files beside `data/outputs` and reads them back. It checks every exported table, the scored polygon attributes, registered attribute tables, database integrity, geometry bytes, feature IDs, the spatial index, and unchanged input-file hashes.

Only after these checks pass does the notebook replace the output directory. The prior directory is kept as a temporary backup during replacement and restored if replacement fails. Files unrelated to this notebook are preserved. Run only one export at a time; on Windows, close the output GeoPackage in GIS software if it is locked.

The final message reports numerical QC and identifies any BSRs whose source crosswalk status is provisional. A provisional source match can coexist with a computationally correct result.

In [11]:
def read_csv_for_comparison(path, expected):
    """Preserve text identifiers and blank audit notes during CSV readback."""
    text_columns = {
        column: str for column in expected.columns
        if pd.api.types.is_object_dtype(expected[column].dtype)
        or pd.api.types.is_string_dtype(expected[column].dtype)
    }
    loaded = pd.read_csv(path, dtype=text_columns)
    for column in text_columns:
        if expected[column].notna().all():
            loaded[column] = loaded[column].fillna("")
    return loaded


def spatial_index_snapshot(path, feature_table, geometry_field):
    prefix = f"rtree_{feature_table}_{geometry_field}"
    with closing(sqlite3.connect(path)) as connection:
        exists = connection.execute("SELECT 1 FROM sqlite_master WHERE type='table' AND name=?", (prefix,)).fetchone()
        rows = connection.execute(f"SELECT * FROM {quote_identifier(prefix)} ORDER BY id").fetchall() if exists else None
        triggers = sorted(
            (name, sql) for name, sql in connection.execute(
                "SELECT name, sql FROM sqlite_master WHERE type='trigger' AND tbl_name=?", (feature_table,)
            ) if name.startswith(prefix + "_")
        )
    return rows, triggers


def publish_output_files(staged_directory, target_directory):
    """Replace only files staged by this notebook, with automatic rollback."""
    relative_files = sorted(
        (
            path.relative_to(staged_directory)
            for path in staged_directory.rglob("*")
            if path.is_file()
        ),
        key=lambda path: path.as_posix().casefold(),
    )
    if not relative_files:
        raise RuntimeError("No staged output files were available to publish.")

    target_directory.mkdir(parents=True, exist_ok=True)
    backup_directory = staged_directory.parent / "previous"
    previous_files = set()

    # Back up only files owned by this notebook. Existing maps and other
    # unrelated outputs are neither copied nor moved.
    for relative_path in relative_files:
        target_path = target_directory / relative_path
        if target_path.exists():
            if not target_path.is_file():
                raise IsADirectoryError(
                    f"Expected an output file but found a directory: {target_path}"
                )
            backup_path = backup_directory / relative_path
            backup_path.parent.mkdir(parents=True, exist_ok=True)
            try:
                shutil.copy2(target_path, backup_path)
            except PermissionError as error:
                raise PermissionError(
                    f"Could not back up {target_path}. Close it in QGIS, "
                    "ArcGIS, Excel, or another application, then rerun this cell."
                ) from error
            previous_files.add(relative_path)

    def replace_from_copy(source_path, target_path, suffix):
        target_path.parent.mkdir(parents=True, exist_ok=True)
        temporary_path = target_path.with_name(
            f".{target_path.name}.{uuid4().hex[:8]}.{suffix}"
        )
        try:
            shutil.copy2(source_path, temporary_path)
            temporary_path.replace(target_path)
        finally:
            try:
                temporary_path.unlink(missing_ok=True)
            except OSError:
                pass

    published = []
    current_relative = None
    try:
        for current_relative in relative_files:
            replace_from_copy(
                staged_directory / current_relative,
                target_directory / current_relative,
                "tmp",
            )
            published.append(current_relative)
    except Exception as publish_error:
        rollback_errors = []
        for relative_path in reversed(published):
            target_path = target_directory / relative_path
            try:
                if relative_path in previous_files:
                    replace_from_copy(
                        backup_directory / relative_path,
                        target_path,
                        "rollback",
                    )
                else:
                    target_path.unlink(missing_ok=True)
            except Exception as rollback_error:
                rollback_errors.append(
                    f"{relative_path}: {rollback_error}"
                )

        failed_path = target_directory / current_relative
        if rollback_errors:
            raise RuntimeError(
                f"Publishing failed at {failed_path}, and rollback was incomplete: "
                + "; ".join(rollback_errors)
            ) from publish_error
        if isinstance(publish_error, PermissionError):
            raise PermissionError(
                f"Could not replace {failed_path}. Existing notebook outputs "
                "were restored. Close the file in QGIS, ArcGIS, Excel, or "
                "another application, then rerun this cell."
            ) from publish_error
        raise RuntimeError(
            f"Publishing failed at {failed_path}; existing notebook outputs "
            "were restored."
        ) from publish_error


check(calculation_qc_passed, "Calculation QC completed", stage="staging")
check(worked_example_qc_passed, "Worked example completed", stage="staging")
check(scoring_state_signature() == validated_calculation_signature, "Scoring tables unchanged since validation", "Run the notebook from the beginning after editing inputs or calculations.", stage="staging")
# Recheck source hashes immediately before export to detect mid-run changes.
for name, path in {**INPUT_PATHS, "spatial": BSR_INPUT_PATH}.items():
    check(file_sha256(path) == input_file_hashes[name], f"Input unchanged: {path.name}", stage="staging")

OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)
with TemporaryDirectory(prefix="atlas_export_") as temporary_root:
    staged = Path(temporary_root) / "outputs"
    staged.mkdir()
    staged_qc = staged / "QC"
    staged_qc.mkdir(exist_ok=True)

    for name, table in CORE_OUTPUTS.items():
        table.to_csv(staged / name, index=False)
    for name, table in QC_OUTPUTS.items():
        table.to_csv(staged_qc / name, index=False)

    staged_gpkg = staged / BSR_OUTPUT_FILE
    feature_table, geometry_field, source_key, score_fields, attribute_names = write_scored_bsr_gpkg(
        BSR_INPUT_PATH, staged_gpkg, bsr_scores, GPKG_ATTRIBUTE_TABLES
    )

    # Check values in every staged core and supporting CSV, not just its existence.
    for directory, tables in [(staged, CORE_OUTPUTS), (staged_qc, QC_OUTPUTS)]:
        for name, expected in tables.items():
            actual = read_csv_for_comparison(directory / name, expected)
            compare_frames(actual, expected, f"CSV readback: {name}", stage="staged output")

    with closing(sqlite3.connect(staged_gpkg)) as connection:
        check(connection.execute("PRAGMA integrity_check").fetchone()[0] == "ok", "GeoPackage database integrity", stage="staged output")
        check(not connection.execute("PRAGMA foreign_key_check").fetchall(), "GeoPackage foreign-key integrity", stage="staged output")
        query = f"SELECT {', '.join(quote_identifier(column) for column in score_fields)} FROM {quote_identifier(feature_table)} ORDER BY score_bsr"
        actual = pd.read_sql_query(query, connection)
        expected = bsr_scores.rename(columns={"bsr": "score_bsr"})[score_fields].sort_values("score_bsr")
        compare_frames(actual, expected, "GeoPackage BSR scores and review fields", stage="staged output")
        registered = {row[0] for row in connection.execute("SELECT table_name FROM gpkg_contents WHERE data_type='attributes'")}
        check(set(attribute_names).issubset(registered), "GeoPackage attribute tables registered", stage="staged output")
        for name, specification in GPKG_ATTRIBUTE_TABLES.items():
            expected = specification["table"]
            columns = ", ".join(quote_identifier(column) for column in expected.columns)
            actual = pd.read_sql_query(f"SELECT {columns} FROM {quote_identifier(name)} ORDER BY fid", connection)
            compare_frames(actual, expected, f"GeoPackage attribute values: {name}", stage="staged output")
        null_geometry = connection.execute(f"SELECT COUNT(*) FROM {quote_identifier(feature_table)} WHERE {quote_identifier(geometry_field)} IS NULL").fetchone()[0]
        check(null_geometry == 0, "No null scored geometry", stage="staged output")
        feature_id = next(row[1] for row in connection.execute(f"PRAGMA table_info({quote_identifier(feature_table)})") if row[5] == 1)

    check(geometry_digest(BSR_INPUT_PATH, feature_table, source_key, geometry_field) == geometry_digest(staged_gpkg, feature_table, source_key, geometry_field), "Geometry and BSR identifiers preserved byte-for-byte", stage="staged output")
    check(geometry_digest(BSR_INPUT_PATH, feature_table, feature_id, geometry_field) == geometry_digest(staged_gpkg, feature_table, feature_id, geometry_field), "Geometry and feature IDs preserved byte-for-byte", stage="staged output")
    check(spatial_index_snapshot(BSR_INPUT_PATH, feature_table, geometry_field) == spatial_index_snapshot(staged_gpkg, feature_table, geometry_field), "Spatial-index rows and trigger definitions preserved", stage="staged output")

    # Bind this output set to the exact input versions used in the calculation.
    for name, path in {**INPUT_PATHS, "spatial": BSR_INPUT_PATH}.items():
        check(file_sha256(path) == input_file_hashes[name], f"Input still unchanged: {path.name}", stage="staged output")
    run_metadata["numerical_qc"] = "input, transformation, calculation, example, and staged-file checks passed"
    metadata_path = staged_qc / "scoring_run_metadata.json"
    metadata_path.write_text(json.dumps(run_metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    check(json.loads(metadata_path.read_text(encoding="utf-8")) == run_metadata, "Run metadata readback", stage="staged output")
    # Write the final QC record after all substantive staging checks are complete.
    qc_summary = pd.DataFrame(qc_records)
    qc_summary.to_csv(staged_qc / "qc_summary.csv", index=False)
    pd.testing.assert_frame_equal(pd.read_csv(staged_qc / "qc_summary.csv"), qc_summary)
    publish_output_files(staged, OUTPUT_DIR)

print(f"Published validated outputs to: {OUTPUT_DIR}")
print(f"Run: {RUN_ID}; numerical and file QC checks passed: {len(qc_summary)}.")
print(f"Source BSR status: {run_metadata['run_review_status']}.")
if unresolved_bsrs:
    print("Source BSR correspondence is provisional: " + ", ".join(unresolved_bsrs))
print("Source condition and vulnerability flags remain documented; numerical QC does not resolve them.")
display(pd.DataFrame([{"file": name, "rows": len(table)} for name, table in CORE_OUTPUTS.items()]))

Published validated outputs to: C:\Users\AlexThornton-Dunwood\OneDrive - Lichen Land & Water\Documents\GitHub\atlas_lvl1-2_scoring_webapp\data\outputs
Run: 20260916T225423Z_9700d92a; numerical and file QC checks passed: 215.
Source BSR status: provisional BSR correspondence.
Source BSR correspondence is provisional: CC1, CC2, CC3, CC4, CC5, CC6, CC7, CC8, CC9
Source condition and vulnerability flags remain documented; numerical QC does not resolve them.


,file,rows
0,bsr_scores.csv,29
1,fish_use_scores.csv,290
2,population_scores.csv,20
3,life_stage_scores.csv,290
4,species_scores.csv,87
5,limiting_factor_scores_integrated.csv,435
6,action_scores.csv,319
7,calculation_grid.csv,4350
